<a href="https://colab.research.google.com/github/gazza-bot/KERIS-Analisis-Sentimen-User-Review/blob/main/Data_Cleansing_Kopken_Review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load CSV

In [2]:
import os
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from google.colab import drive

# initialize drive location
drive.mount('/content/drive')
folder_path = '/content/drive/MyDrive/KERIS-Coba/Scrap-Kopken'
file_path = os.path.join(folder_path, 'HIGH_Resto_Kopken_review.csv')

# Download req NLTK data
nltk.download("stopwords")
nltk.download("wordnet")

# Load Scraped data
df = pd.read_csv(file_path)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,8716a7a8-2378-42a1-9cc5-1ac4a8671f83,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"pass dilidah, mantab kopinya",5,0,126.10.08,2026-10-21 05:38:51,Hi Kak 😊\nTerima kasih atas review baiknya unt...,2026-10-21 05:41:19,126.10.08
1,72864359-69fb-4b35-b92c-85051ea5eed3,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,mantab,5,0,126.10.08,2026-10-21 05:05:46,"Hi Kak, terima kasih atas review baiknya untuk...",2026-10-21 05:07:11,126.10.08
2,d6207072-64ad-402b-9c55-0f116a8b9ece,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,best coffee,5,0,126.10.08,2026-10-21 04:38:23,Hi Kak 😊\nTerima kasih atas review baiknya unt...,2026-10-21 04:39:19,126.10.08
3,048a8432-b299-49c2-9650-06bec417c8d5,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,kopi enak,5,0,126.10.08,2026-10-21 03:35:26,"Hi Kak, terima kasih atas review baiknya untuk...",2026-10-21 03:37:20,126.10.08
4,b3cf67d3-e9f1-413f-88a1-fc0b9a538ac0,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,One of my fav coffee brand,5,0,126.10.08,2026-10-21 03:00:44,Hi Kak 😊\nTerima kasih atas review baiknya unt...,2026-10-21 03:01:32,126.10.08


In [3]:
#  drop columns that unused
drop_columns = ['userName', 'userImage', 'reviewCreatedVersion', 'repliedAt', 'at']
df = df.drop(columns=drop_columns, errors='ignore')

df.head()

,reviewId,content,score,thumbsUpCount,replyContent,appVersion
0,8716a7a8-2378-42a1-9cc5-1ac4a8671f83,"pass dilidah, mantab kopinya",5,0,Hi Kak 😊\nTerima kasih atas review baiknya unt...,126.10.08
1,72864359-69fb-4b35-b92c-85051ea5eed3,mantab,5,0,"Hi Kak, terima kasih atas review baiknya untuk...",126.10.08
2,d6207072-64ad-402b-9c55-0f116a8b9ece,best coffee,5,0,Hi Kak 😊\nTerima kasih atas review baiknya unt...,126.10.08
3,048a8432-b299-49c2-9650-06bec417c8d5,kopi enak,5,0,"Hi Kak, terima kasih atas review baiknya untuk...",126.10.08
4,b3cf67d3-e9f1-413f-88a1-fc0b9a538ac0,One of my fav coffee brand,5,0,Hi Kak 😊\nTerima kasih atas review baiknya unt...,126.10.08


Missing Values and Duplicate

In [4]:
missing_values = df['content'].isna().sum()
print(f"Jumlah Missing values : {missing_values}")

duplicate_values = df.duplicated(subset=['content']).sum()
print(f"Jumlah data yang terduplikat : {duplicate_values}")

Jumlah Missing values : 0
Jumlah data yang terduplikat : 11348


In [5]:
# Jumlah data sebelum di drop duplikat
print(f"Jumlah data (Sebelum drop): {df.shape[0]}")
# Drop cells duplicate
df = df.drop_duplicates(subset=['content'], keep='first')
print("Data duplikat berhasil dihapus")

# Jumlah data
print(f"Jumlah data (Setelah drop): {df.shape[0]}")

Jumlah data (Sebelum drop): 36248
Data duplikat berhasil dihapus
Jumlah data (Setelah drop): 24900


Normalisasi Teks Komen

In [6]:
pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.5 MB/s eta 0:00:00


In [32]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# 1. Stopwords: jaga kata negasi & penguat
stop_words = StopWordRemoverFactory().get_stop_words()

jaga = {'tidak', 'tak', 'bukan', 'belum', 'kurang', 'jangan', 'sangat', 'lain'}
stop_words = [w for w in stop_words if w not in jaga]
stop_words.extend(['nya', 'sih', 'dong', 'deh', 'ya', 'kan'])
stop_words = set(stop_words)  # set lebih cepat

# 2. Kamus slang (tambahkan sesuai temuan di datamu)
slang_dict = {
    "bgs": "bagus", "bgt": "banget", "gk": "tidak", "ga": "tidak", "gak": "tidak",
    "apk": "aplikasi", "sy": "saya", "klo": "kalau", "udh": "sudah",
    "tp": "tapi", "dpt": "dapat", "knp": "kenapa", "yg": "yang", "gmn": "bagaimana",
    "mantab": "mantap", "rekomended": "direkomendasikan", "pass": "pas",
    "dilidah": "di lidah", "mw" : "mau", "kalo": "kalau", "udah": "sudah",
    "pake": "pakai","dapet": "dapat","jd": "jadi","pesen": "pesan",
    "app": "aplikasi", "thn": "tahun","dgn": "dengan","ny": "nya","inget": "ingat",
    "recomend": "rekomendasi",
    "endul": "enak","mantul": "mantap","n": "dan"
}

def normalisasi_teks(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'(.)\1{1,}', r'\1', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    words = text.split()
    words = [slang_dict.get(w, w) for w in words]
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

# 3. Terapkan
df['clean_data'] = df['content'].apply(normalisasi_teks)
df['clean_data'] = df['clean_data'].replace('', pd.NA)
df = df.dropna(subset=['clean_data']).copy()

# 4. Label sentimen dari rating
def beri_label(score):
    if score <= 2:
        return 'negatif'
    elif score == 3:
        return 'netral'
    return 'positif'

df['label'] = df['score'].apply(beri_label)

# 5. Cek hasil
print(df[['content', 'clean_data', 'score', 'label']].head())
print("Value Counts : ", df['label'].value_counts())

# 6. Simpan (kolom content asli tetap ada)
df.to_csv('/content/drive/MyDrive/KERIS-Coba/Scrap-Kopken/cleaned_app_reviews_final.csv', index=False)
print("Pembersihan selesai dan data telah disimpan!")

                        content                   clean_data  score    label
0  pass dilidah, mantab kopinya  pas di lidah mantap kopinya      5  positif
1                        mantab                       mantap      5  positif
2                   best coffee                    best cofe      5  positif
3                     kopi enak                    kopi enak      5  positif
4    One of my fav coffee brand     one of my fav cofe brand      5  positif
Value Counts :  label
positif    23239
negatif     1225
netral       226
Name: count, dtype: int64
Pembersihan selesai dan data telah disimpan!


In [28]:
# cari kata terbanyak setelah dilakukan cleansing
from collections import Counter
print(Counter(" ".join(df['clean_data']).split()).most_common(100))

[('kopi', 6987), ('kenangan', 3240), ('aplikasi', 3172), ('mantap', 3150), ('enak', 3045), ('tidak', 2924), ('banget', 2811), ('promo', 2623), ('banyak', 2289), ('sangat', 1887), ('kopinya', 1811), ('bagus', 1611), ('suka', 1593), ('rasa', 1474), ('pelayanan', 1439), ('mudah', 1249), ('jadi', 1130), ('lebih', 1032), ('kalau', 1023), ('buat', 1018), ('selalu', 968), ('aplikasinya', 957), ('ramah', 950), ('promonya', 900), ('cepat', 869), ('pesan', 861), ('harga', 858), ('beli', 857), ('pakai', 857), ('sama', 807), ('rasanya', 803), ('membantu', 802), ('mantan', 794), ('pas', 783), ('god', 782), ('kopken', 696), ('best', 691), ('oke', 682), ('baik', 674), ('diskon', 667), ('keren', 631), ('sering', 619), ('pelayananya', 551), ('aja', 521), ('order', 505), ('terus', 495), ('the', 488), ('pokoknya', 481), ('memudahkan', 476), ('outlet', 473), ('sekali', 467), ('mau', 463), ('aku', 461), ('perlu', 453), ('bikin', 447), ('antri', 446), ('nyaman', 442), ('tempat', 435), ('baru', 435), ('lah',

In [31]:
print(df[df['clean_data'].str.contains(r'\bmantan\b')][['content', 'clean_data']].head(10))

                                                content  \
163     8 thn gak pernah berpaling dari kenangan mantan   
261   dah ga mw ke lain hati selain kopi kenangan ma...   
586                          kopi kenangan mantan endul   
814                                  mantan ny recomend   
869   kopi kenangan mantan jadi favorit di keluarga,...   
969                 Setiap hari ngopi kenangan mantan 🥰   
1032                   INGET MANTAN AKU PASTI KESINI :(   
1054               selalu suka dgn kopi kenangan mantan   
1226               kopi kenangan mantan bikin ketagihan   
1248                            kenangan mantan favorit   

                                             clean_data  
163        tahun tidak pernah berpaling kenangan mantan  
261   dah tidak mau lain hati kopi kenangan mantan p...  
586                           kopi kenangan mantan enak  
814                                  mantan rekomendasi  
869   kopi kenangan mantan jadi favorit keluarga jag...  
96